In [ ]:
# Import necessary libraries
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

# Step 1: Load user and video data (assumes CSVs but can replace with database queries)
# Replace these with your actual database calls
user_history = pd.read_csv("user_history.csv")  # User interaction history: user_id, video_id, interaction_score
video_metadata = pd.read_csv("video_metadata.csv")  # Video info: video_id, title, tags, category

# Preview the data
print("User History:\n", user_history.head())
print("Video Metadata:\n", video_metadata.head())

# Step 2: Content-based filtering
def content_based_recommendations(user_id, user_history, video_metadata, top_n=5):
    # Merge user history with video metadata
    user_videos = user_history[user_history['user_id'] == user_id]
    user_videos = user_videos.merge(video_metadata, on='video_id', how='inner')
    
    # Combine tags and category as a single content feature
    video_metadata['content'] = video_metadata['tags'] + " " + video_metadata['category']
    
    # TF-IDF Vectorizer for text similarity
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(video_metadata['content'])
    
    # Get user interaction video IDs and calculate similarity scores
    user_video_ids = user_videos['video_id'].tolist()
    content_similarities = cosine_similarity(tfidf_matrix)
    
    # Average content similarity for non-watched videos
    candidate_videos = video_metadata[~video_metadata['video_id'].isin(user_video_ids)]
    candidate_indices = candidate_videos.index
    avg_similarity_scores = content_similarities[user_video_ids].mean(axis=0)
    
    # Get top recommendations
    recommended_indices = avg_similarity_scores.argsort()[-top_n:][::-1]
    return video_metadata.iloc[recommended_indices][['video_id', 'title']]

# Step 3: Collaborative filtering
def collaborative_recommendations(user_id, user_history, video_metadata, top_n=5):
    # Create user-item interaction matrix
    user_item_matrix = user_history.pivot_table(index='user_id', columns='video_id', values='interaction_score', fill_value=0)
    user_item_sparse = csr_matrix(user_item_matrix.values)
    
    # Fit kNN model for collaborative filtering
    model_knn = NearestNeighbors(metric='cosine', algorithm='brute')
    model_knn.fit(user_item_sparse)
    
    # Find nearest neighbors
    user_index = user_item_matrix.index.tolist().index(user_id)
    distances, indices = model_knn.kneighbors(user_item_sparse[user_index], n_neighbors=top_n+1)
    
    # Recommend videos from similar users
    similar_users = indices.flatten()[1:]  # Exclude self
    similar_users_videos = user_history[user_history['user_id'].isin(similar_users)]['video_id']
    recommended_videos = video_metadata[video_metadata['video_id'].isin(similar_users_videos)]
    
    return recommended_videos[['video_id', 'title']].head(top_n)

# Step 4: Hybrid recommendation system
def hybrid_recommendations(user_id, user_history, video_metadata, top_n=5):
    content_recs = content_based_recommendations(user_id, user_history, video_metadata, top_n)
    collaborative_recs = collaborative_recommendations(user_id, user_history, video_metadata, top_n)
    
    # Combine recommendations with weights (e.g., 70% content, 30% collaborative)
    combined_recs = pd.concat([content_recs, collaborative_recs]).drop_duplicates().head(top_n)
    return combined_recs

# Test the recommendation system
user_id = 1  # Replace with your test user ID
recommendations = hybrid_recommendations(user_id, user_history, video_metadata, top_n=5)
print("\nRecommended Videos for User {}:\n".format(user_id), recommendations)

